<a href="https://colab.research.google.com/github/mbahramii/conveyor-stone-detector/blob/main/Stone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile('/content/frames.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ Done!")

In [ ]:
import os
files = os.listdir('/content/frames/')
print(f"تعداد: {len(files)}")
print("نمونه:", sorted(files)[:5])


In [ ]:
import os
print(os.listdir('/content/'))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

class StoneDetector:
    def __init__(self):
        self.clahe_clip_limit = 2.0
        self.clahe_tile_size = (8, 8)
        self.blur_kernel = 5
        self.adaptive_block_size = 51
        self.adaptive_C = 10
        self.morph_open_iter = 2
        self.morph_close_iter = 3
        self.morph_open_kernel_size = (3, 3)
        self.morph_close_kernel_size = (9, 9)
        self.min_stone_area = 300
        self.max_stone_area_ratio = 0.6
        self.min_solidity = 0.3
        self.min_aspect_ratio = 0.15
        self.max_aspect_ratio = 8.0

    def define_detection_zone(self, h, w):
          return np.array([
              [int(w * 0.62), int(h * 0.64)],
              [int(w * 0.98), int(h * 0.64)],
              [int(w * 0.98), int(h * 0.98)],
              [int(w * 0.62), int(h * 0.98)],
          ], dtype=np.int32)
    def create_roi_mask(self, h, w, roi_points):
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [roi_points], 255)
        return mask

    def detect_stones(self, frame):
        h, w = frame.shape[:2]
        roi_points = self.define_detection_zone(h, w)
        roi_mask   = self.create_roi_mask(h, w, roi_points)

        gray     = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        clahe    = cv2.createCLAHE(clipLimit=self.clahe_clip_limit,
                                    tileGridSize=self.clahe_tile_size)
        enhanced = clahe.apply(gray)
        masked   = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)
        blurred  = cv2.GaussianBlur(masked, (self.blur_kernel, self.blur_kernel), 0)

        binary = cv2.adaptiveThreshold(
            blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, self.adaptive_block_size, self.adaptive_C)
        binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, self.morph_close_kernel_size)
        k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, self.morph_open_kernel_size)
        k_dil   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=self.morph_close_iter)
        binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=self.morph_open_iter)
        binary  = cv2.dilate(binary, k_dil, iterations=1)

        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        max_area = h * w * self.max_stone_area_ratio
        stones = []
        for cnt in cnts:
            area = cv2.contourArea(cnt)
            if area < self.min_stone_area or area > max_area:
                continue
            hull      = cv2.convexHull(cnt)
            hull_area = cv2.contourArea(hull)
            if hull_area == 0:
                continue
            if area / hull_area < self.min_solidity:
                continue
            x, y, bw, bh = cv2.boundingRect(cnt)
            ar = float(bw) / (bh + 1e-5)
            if ar < self.min_aspect_ratio or ar > self.max_aspect_ratio:
                continue
            stones.append({'contour': cnt, 'bbox': (x, y, bw, bh), 'area': area})

        result = frame.copy()
        cv2.polylines(result, [roi_points], True, (0, 255, 255), 2)
        for i, s in enumerate(stones):
            x, y, bw, bh = s['bbox']
            cv2.rectangle(result, (x, y), (x+bw, y+bh), (255, 0, 0), 2)
            cv2.drawContours(result, [s['contour']], -1, (255, 0, 0), 2)
            cv2.putText(result, f"Stone {i+1}", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.putText(result, f"Stones: {len(stones)}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

        enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
        mask_bgr     = cv2.cvtColor(binary,   cv2.COLOR_GRAY2BGR)
        cv2.polylines(mask_bgr, [roi_points], True, (0, 255, 255), 2)

        return frame, enhanced_bgr, mask_bgr, result, len(stones)



FRAMES_DIR = "/content/frames"
ff = sorted(Path(FRAMES_DIR).glob("*.jpg"))[149]
img = cv2.imread(str(ff))

detector = StoneDetector()
original, clahe_img, mask, result, count = detector.detect_stones(img)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes[0,0].imshow(cv2.cvtColor(original,  cv2.COLOR_BGR2RGB)); axes[0,0].set_title("Original");        axes[0,0].axis('off')
axes[0,1].imshow(cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)); axes[0,1].set_title("CLAHE Enhanced");  axes[0,1].axis('off')
axes[1,0].imshow(cv2.cvtColor(mask,      cv2.COLOR_BGR2RGB)); axes[1,0].set_title("Detection Mask");  axes[1,0].axis('off')
axes[1,1].imshow(cv2.cvtColor(result,    cv2.COLOR_BGR2RGB)); axes[1,1].set_title(f"Result (Stones: {count})"); axes[1,1].axis('off')
plt.tight_layout()
plt.show()
print(f"✅ Stones detected: {count}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

class StoneDetector:
    def __init__(self):
        self.clahe_clip_limit = 2.0
        self.clahe_tile_size = (8, 8)
        self.blur_kernel = 5
        self.adaptive_block_size = 51
        self.adaptive_C = 10
        self.morph_open_iter = 2
        self.morph_close_iter = 3
        self.morph_open_kernel_size = (3, 3)
        self.morph_close_kernel_size = (9, 9)
        self.min_stone_area = 300
        self.max_stone_area_ratio = 0.6
        self.min_solidity = 0.3
        self.min_aspect_ratio = 0.15
        self.max_aspect_ratio = 8.0

    def define_detection_zone(self, h, w):
        return np.array([
              [int(w * 0.62), int(h * 0.64)],   # بالا-چپ
              [int(w * 0.98), int(h * 0.64)],   # بالا-راست
              [int(w * 0.98), int(h * 0.98)],   # پایین-راست
              [int(w * 0.62), int(h * 0.98)],   # پایین-چپ
        ], dtype=np.int32)

    def create_roi_mask(self, h, w, roi_points):
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [roi_points], 255)
        return mask

    def detect_stones(self, frame):
        h, w = frame.shape[:2]
        roi_points = self.define_detection_zone(h, w)
        roi_mask   = self.create_roi_mask(h, w, roi_points)

        gray     = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        clahe    = cv2.createCLAHE(clipLimit=self.clahe_clip_limit,
                                    tileGridSize=self.clahe_tile_size)
        enhanced = clahe.apply(gray)
        masked   = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)
        blurred  = cv2.GaussianBlur(masked, (self.blur_kernel, self.blur_kernel), 0)

        binary = cv2.adaptiveThreshold(
            blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, self.adaptive_block_size, self.adaptive_C)


        binary = cv2.bitwise_and(binary, binary, mask=roi_mask)


        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, self.morph_close_kernel_size)
        k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, self.morph_open_kernel_size)
        k_dil   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=self.morph_close_iter)
        binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=self.morph_open_iter)
        binary  = cv2.dilate(binary, k_dil, iterations=1)

        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        max_area = h * w * self.max_stone_area_ratio

        valid = []
        for cnt in cnts:
            area = cv2.contourArea(cnt)
            if area < self.min_stone_area or area > max_area:
                continue
            hull      = cv2.convexHull(cnt)
            hull_area = cv2.contourArea(hull)
            if hull_area == 0:
                continue
            if area / hull_area < self.min_solidity:
                continue
            x, y, bw, bh = cv2.boundingRect(cnt)
            ar = float(bw) / (bh + 1e-5)
            if ar < self.min_aspect_ratio or ar > self.max_aspect_ratio:
                continue
            valid.append(cnt)

        result = frame.copy()
        cv2.polylines(result, [roi_points], True, (0, 255, 255), 2)

        stone_count = 0
        if valid:

            best = max(valid, key=cv2.contourArea)
            rect = cv2.minAreaRect(best)
            box  = np.intp(cv2.boxPoints(rect))
            cv2.drawContours(result, [box], 0, (255, 0, 0), 2)
            cx, cy = int(rect[0][0]), int(rect[0][1])
            cv2.circle(result, (cx, cy), 5, (255, 0, 0), -1)
            stone_count = 1

        cv2.putText(result, f"Stones: {stone_count}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

        enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
        mask_bgr     = cv2.cvtColor(binary,   cv2.COLOR_GRAY2BGR)
        cv2.polylines(mask_bgr, [roi_points], True, (0, 255, 255), 2)

        return frame, enhanced_bgr, mask_bgr, result, stone_count


FRAMES_DIR = "/content/frames"
ff = sorted(Path(FRAMES_DIR).glob("*.jpg"))[149]
img = cv2.imread(str(ff))

detector = StoneDetector()
original, clahe_img, mask, result, count = detector.detect_stones(img)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes[0,0].imshow(cv2.cvtColor(original,  cv2.COLOR_BGR2RGB)); axes[0,0].set_title("Original");       axes[0,0].axis('off')
axes[0,1].imshow(cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)); axes[0,1].set_title("CLAHE");          axes[0,1].axis('off')
axes[1,0].imshow(cv2.cvtColor(mask,      cv2.COLOR_BGR2RGB)); axes[1,0].set_title("Detection Mask"); axes[1,0].axis('off')
axes[1,1].imshow(cv2.cvtColor(result,    cv2.COLOR_BGR2RGB)); axes[1,1].set_title(f"Result (Stones: {count})"); axes[1,1].axis('off')
plt.tight_layout()
plt.show()
print(f"✅ Stones: {count}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FRAMES_DIR = "/content/frames"
frame_files = sorted(Path(FRAMES_DIR).glob("*.jpg")) + \
              sorted(Path(FRAMES_DIR).glob("*.png"))

ff  = frame_files[145]
img = cv2.imread(str(ff))
h, w = img.shape[:2]

# zone
zone = np.array([
    [int(w * 0.62), int(h * 0.64)],
    [int(w * 0.98), int(h * 0.64)],
    [int(w * 0.98), int(h * 0.98)],
    [int(w * 0.62), int(h * 0.98)],
], dtype=np.int32)

# mask zone
roi_mask = np.zeros((h, w), dtype=np.uint8)
cv2.fillPoly(roi_mask, [zone], 255)

# CLAHE
gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(gray)

# blur
blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)

# adaptive threshold
binary = cv2.adaptiveThreshold(
    blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV, 51, 10)


binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

# morphology
k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=3)
binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=2)

binary_inv = cv2.bitwise_not(binary)
binary_inv = cv2.bitwise_and(binary_inv, binary_inv, mask=roi_mask)

cnts, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

vis = img.copy()
cv2.polylines(vis, [zone], True, (0, 255, 255), 2)

if cnts:
    best = max(cnts, key=cv2.contourArea)
    rect = cv2.minAreaRect(best)
    box  = np.intp(cv2.boxPoints(rect))
    cv2.drawContours(vis, [box], 0, (255, 0, 0), 2)
    cv2.putText(vis, "Stone", (box[0][0], box[0][1]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(img,    cv2.COLOR_BGR2RGB)); axes[0].set_title("Original");  axes[0].axis('off')
axes[1].imshow(binary, cmap='gray');                     axes[1].set_title("Mask");       axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(vis,    cv2.COLOR_BGR2RGB)); axes[2].set_title("Result");     axes[2].axis('off')
plt.tight_layout()
plt.show()

In [1]:
import cv2
import numpy as np
from pathlib import Path

INPUT_VIDEO  = "/content/input.mp4"
OUTPUT_VIDEO = "/content/output.mp4"

cap = cv2.VideoCapture(INPUT_VIDEO)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

print(f"✅ ویدیو: {width}x{height} | {fps}fps | {total} فریم")

frame_count = 0
while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    h, w = img.shape[:2]

    # zone
    zone = np.array([
        [int(w * 0.62), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.98)],
        [int(w * 0.62), int(h * 0.98)],
    ], dtype=np.int32)

    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(roi_mask, [zone], 255)

    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    blurred  = cv2.GaussianBlur(enhanced, (5, 5), 0)

    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 51, 10)
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=3)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=2)

    binary_inv = cv2.bitwise_not(binary)
    binary_inv = cv2.bitwise_and(binary_inv, binary_inv, mask=roi_mask)

    cnts, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

    vis = img.copy()
    cv2.polylines(vis, [zone], True, (0, 255, 255), 2)

    stone_found = False
    if cnts:
        best = max(cnts, key=cv2.contourArea)
        if cv2.contourArea(best) > 1000:
            rect = cv2.minAreaRect(best)
            box  = np.intp(cv2.boxPoints(rect))
            cv2.drawContours(vis, [box], 0, (255, 0, 0), 2)
            cv2.putText(vis, "Stone", (box[0][0], box[0][1]-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            stone_found = True

    status = "STONE ✓" if stone_found else "NO STONE"
    color  = (0, 255, 0) if stone_found else (0, 0, 255)
    cv2.putText(vis, status, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

    out.write(vis)
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"  ⏳ {frame_count}/{total} فریم پردازش شد...")

cap.release()
out.release()
print(f"\n✅ ویدیو ذخیره شد: {OUTPUT_VIDEO}")

✅ ویدیو: 848x526 | 20fps | 1572 فریم
  ⏳ 30/1572 فریم پردازش شد...
  ⏳ 60/1572 فریم پردازش شد...
  ⏳ 90/1572 فریم پردازش شد...
  ⏳ 120/1572 فریم پردازش شد...
  ⏳ 150/1572 فریم پردازش شد...
  ⏳ 180/1572 فریم پردازش شد...
  ⏳ 210/1572 فریم پردازش شد...
  ⏳ 240/1572 فریم پردازش شد...
  ⏳ 270/1572 فریم پردازش شد...
  ⏳ 300/1572 فریم پردازش شد...
  ⏳ 330/1572 فریم پردازش شد...
  ⏳ 360/1572 فریم پردازش شد...
  ⏳ 390/1572 فریم پردازش شد...
  ⏳ 420/1572 فریم پردازش شد...
  ⏳ 450/1572 فریم پردازش شد...
  ⏳ 480/1572 فریم پردازش شد...
  ⏳ 510/1572 فریم پردازش شد...
  ⏳ 540/1572 فریم پردازش شد...
  ⏳ 570/1572 فریم پردازش شد...
  ⏳ 600/1572 فریم پردازش شد...
  ⏳ 630/1572 فریم پردازش شد...
  ⏳ 660/1572 فریم پردازش شد...
  ⏳ 690/1572 فریم پردازش شد...
  ⏳ 720/1572 فریم پردازش شد...
  ⏳ 750/1572 فریم پردازش شد...
  ⏳ 780/1572 فریم پردازش شد...
  ⏳ 810/1572 فریم پردازش شد...
  ⏳ 840/1572 فریم پردازش شد...
  ⏳ 870/1572 فریم پردازش شد...
  ⏳ 900/1572 فریم پردازش شد...
  ⏳ 930/1572 فریم پردازش شد...
  ⏳ 9

In [2]:
from google.colab import files
files.download("/content/output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>